# CPU-only analysis

Purpose: compare CPU inference throughput, latency, memory demand, and scaling behaviour across the four CPU platforms while retaining only rows whose backend is `cpu`.

This notebook reads only `results/processed/final-analysis-dataset.csv` and writes derived figures and tables under `analysis/`. Missing measurements are excluded from the relevant calculation rather than replaced with zero.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "analysis").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.utils import load_dataset, save_figure, save_table, successful, grouped_bar, add_display_hardware

data = load_dataset(PROJECT_ROOT / "results" / "processed" / "final-analysis-dataset.csv")
print(f"Loaded {len(data):,} rows and {len(data.columns):,} columns")

## CPU analysis subset

The analysis uses successful CPU executions for performance summaries. The filter is based on the recorded backend, not on a hardcoded row count.

In [ ]:
cpu = add_display_hardware(data.loc[data["backend"].eq("cpu")].copy())
cpu_success = successful(cpu)
print(f"CPU rows: {len(cpu):,}; successful rows: {len(cpu_success):,}")
summary = cpu_success.groupby(["hardware_label", "model"], as_index=False).agg(decode_tps=("decode_tps", "mean"), prompt_eval_tps=("prompt_eval_tps", "mean"), time_to_first_token=("time_to_first_token", "mean"), total_time=("total_time", "mean"), ram_usage=("ram_usage", "mean"))
save_table(summary, "02_cpu_performance_summary.csv")

## Decode throughput

Decode throughput is the main token-generation speed measure. Means are shown for each CPU and model combination; missing decode measurements are omitted by pandas during aggregation.

In [ ]:
grouped_bar(cpu_success, "hardware_label", "decode_tps", "model", "CPU decode throughput by hardware and model", "Mean decode throughput (tokens/s)", "02_cpu_decode_tps.png", rotate_labels=True)

## Prompt evaluation throughput

Prompt evaluation speed describes prompt-processing performance separately from generated-token decoding.

In [ ]:
grouped_bar(cpu_success, "hardware_label", "prompt_eval_tps", "model", "CPU prompt evaluation throughput", "Mean prompt evaluation throughput (tokens/s)", "02_cpu_prompt_eval_tps.png", rotate_labels=True)

## Time to first token

TTFT is treated as a latency measure, so lower values indicate faster initial response.

In [ ]:
grouped_bar(cpu_success, "hardware_label", "time_to_first_token", "model", "CPU time to first token", "Mean TTFT", "02_cpu_ttft.png", rotate_labels=True)

## Total inference time

Total inference time captures end-to-end generation duration for the recorded workload.

In [ ]:
grouped_bar(cpu_success, "hardware_label", "total_time", "model", "CPU total inference time", "Mean total inference time", "02_cpu_total_time.png", rotate_labels=True)

## RAM usage

RAM usage is compared using available measurements only; absent values remain absent rather than being interpreted as zero.

In [ ]:
grouped_bar(cpu_success, "hardware_label", "ram_usage", "model", "CPU RAM usage", "Mean RAM usage (MB)", "02_cpu_ram_usage.png", rotate_labels=True)

## CPU model scaling

This analysis tests the relationship between nominal dense/MoE model size and CPU decode throughput. Each point is the mean of successful observations for one model.

In [ ]:
scaling = cpu_success.groupby(["model", "model_size_b"], as_index=False).agg(decode_tps=("decode_tps", "mean"))
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(scaling["model_size_b"], scaling["decode_tps"], s=70)
for _, row in scaling.iterrows(): ax.annotate(row["model"], (row["model_size_b"], row["decode_tps"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_title("CPU decode throughput versus model size")
ax.set_xlabel("Model size (B parameters)")
ax.set_ylabel("Mean decode throughput (tokens/s)")
ax.grid(alpha=0.25)
save_figure(fig, "02_cpu_model_scaling.png")
plt.show()
save_table(scaling, "02_cpu_model_scaling.csv")